In [ ]:
import os
import time
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
import torch
from sentence_transformers import SentenceTransformer
import torch.nn.functional as F
import uvicorn
import threading

In [ ]:
class EmbedRequest(BaseModel):
    texts: List[str]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True, device=device)

In [ ]:
app = FastAPI()

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/embed")
def embed(request: EmbedRequest):
    prefixed_texts = ["search_document: " + t for t in request.texts]
    embeddings = model.encode(
        prefixed_texts,
        batch_size=32,
        show_progress_bar=False
    )

    return {"embeddings": embeddings.tolist()}

In [ ]:
def run():
    uvicorn.run(app, host="0.0.0.0", port=8002)

threading.Thread(target=run).start()

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
!cloudflared tunnel --url http://localhost:8002

In [ ]:
!pkill -f uvicorn